In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
needforheat_file_path='needforheat_export_student.parquet'
remeha_file_path='remeha_export_student.parquet'
weather_interpolation_locations_file_path = 'weather_interpolation_locations.parquet'

In [ ]:
%%time
# Attempt to read the Parquet file needforheat
try:
    df_nfh= pd.read_parquet(
        needforheat_file_path,
        engine='pyarrow',
        use_nullable_dtypes=True #Outcommented because of futureWarning
        #dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
%%time
# Attempt to read the Parquet file remeha
try:
    df_remeha = pd.read_parquet(
        remeha_file_path,
        engine='pyarrow',
        use_nullable_dtypes=True  #Outcommented because of futureWarning
        #dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
%%time
# Attempt to read the Parquet file weather
try:
    df_weather = pd.read_parquet(
        weather_interpolation_locations_file_path,
        engine='pyarrow',
        use_nullable_dtypes=True  #Outcommented because of futureWarning
        #dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
#dataframe need for heat
df_nfh

In [ ]:
df_nfh.reset_index(inplace=True)

In [ ]:
df_nfh.info()

In [ ]:
# Extract the date from the timestamp
df_nfh['date'] = df_nfh['timestamp'].dt.date

# Group by ID and date, then calculate the average temperature
average_temp_per_day_per_id = df_nfh.groupby(['id', 'date'])['value'].mean().reset_index()

# Print the resulting DataFrame
print(average_temp_per_day_per_id)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df_nfh is your DataFrame with the provided data
# Extract the date from the timestamp
df_nfh['date'] = df_nfh['timestamp'].dt.date

def plot_average_temp_interactive(df):
    # Filter the DataFrame to include only 'temp_in__degC' property
    temp_df = df[df['property'] == 'temp_in__degC']
    
    # Group by ID and date, then calculate the average temperature
    average_temp_per_day_per_id = temp_df.groupby(['id', 'date'])['value'].mean().reset_index()
    
    # Plot settings
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))
    
    # Plot each ID's average temperature per day
    for id in average_temp_per_day_per_id['id'].unique():
        id_data = average_temp_per_day_per_id[average_temp_per_day_per_id['id'] == id]
        plt.plot(id_data['date'], id_data['value'], label=f'ID {id}')
    
    # Add labels and legend
    plt.title('Average Temperature per ID')
    plt.xlabel('Date')
    plt.ylabel('Average Temperature')
    plt.xticks(rotation=45)
    plt.legend(title='ID', loc='upper left')
    
    # Show plot
    plt.tight_layout()
    plt.show()

# Example usage:
plot_average_temp_interactive(df_nfh)


In [ ]:
import historicdutchweather
from datetime import datetime
import pandas as pd

startdate = datetime(2023, 10, 8)
enddate = datetime.now()

lat, lon = 52.50938258156131, 6.1113935843428715

df = historicdutchweather.get_local_weather(startdate, enddate, lat, lon, metrics=['T'])
df.index = pd.to_datetime(df.index)

# Resample to daily frequency and calculate the mean
daily_avg = df.resample('D').mean()

# Reset index, the index (date) becomes a column, and rename the column to 'date'
daily_avg.reset_index(inplace=True)
daily_avg.rename(columns={'index': 'date'}, inplace=True)
daily_avg['date'] = daily_avg['date'].dt.date

# Rename the 'T' column to 'value'
daily_avg.rename(columns={'T': 'Temp_out'}, inplace=True)

daily_avg.head(10)
#daily_avg.plot();
#plt.title(historicdutchweather.column_descriptions['T']);

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, interact, widgets
import historicdutchweather
from datetime import datetime
import pandas as pd

df_nfh['date'] = df_nfh['timestamp'].dt.date

df_nfh_copy = df_nfh.reset_index().copy()
df_weather_copy = daily_avg.reset_index().copy()

# Make the function for plotting the data. This function gets a copy of the data from remeha as well as a copy of the dutchhistoric weather data. 
# It also gets the standerd data_type of temp_in__degC so the plot starts at showing the temperature side of things. 
# Selected_id ='573579' is the starting value for the plot.
def plot_data(df_nfh_copy, df_weather_copy, data_type='temp_in__degC', selected_id='545968'):
    
    if data_type == 'temp_in__degC':
        filtered_df_house = df_nfh_copy[(df_nfh_copy['property'] == data_type) & (df_nfh_copy['id'] == selected_id)]
        average_per_day_house = filtered_df_house.groupby(['date'])['value'].mean().reset_index()
        
        # Filter df_weather_copy to match the date range of average_per_day_house
        start_date = average_per_day_house['date'].min()
        end_date = average_per_day_house['date'].max()
        df_weather_filtered = df_weather_copy[(df_weather_copy['date'] >= start_date) & (df_weather_copy['date'] <= end_date)]
        
        plt.figure(figsize=(10, 6))
        plt.plot(average_per_day_house['date'], average_per_day_house['value'], marker='o', linestyle='-', color='blue', label=f'House {selected_id} Temperature')
        plt.plot(df_weather_filtered['date'], df_weather_filtered['Temp_out'], linestyle='-', color='red', label='Outside Temperature')
        
        plt.title('Temperature Comparison')
        plt.xlabel('Date')
        plt.ylabel('Temperature')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        
    else:
        if selected_id is None:
            print("Please select a House ID.")
            return

        filtered_df_gas = df_nfh_copy
        filtered_df_gas = filtered_df_gas[(filtered_df_gas['property'] == 'g_use_cum__m3') & (filtered_df_gas['id'] == selected_id)]
        filtered_df_gas['timestamp'] = pd.to_datetime(filtered_df_gas['timestamp'])
        filtered_df_gas = filtered_df_gas.sort_values(by=['timestamp'])
        filtered_df_gas.loc[:, 'gas_usage_diff'] = filtered_df_gas.groupby('id')['value'].diff().fillna(0)
        average_per_day_gas = filtered_df_gas.groupby(['id', filtered_df_gas['timestamp'].dt.date])['gas_usage_diff'].sum().reset_index()
        
        plt.figure(figsize=(10, 6))
        plt.plot(average_per_day_gas['timestamp'], average_per_day_gas['gas_usage_diff'], marker='o', linestyle='-', color='blue')
        title = 'Daily Gas Use (kWh) for House '
        plt.title(title + str(selected_id))
        plt.xlabel('Date')
        plt.ylabel('Gas Use (kWh)')
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        plt.show()


# Adding toggle button to switch between the data_types for temperature and gas usage
data_type_selector = widgets.ToggleButtons(
    options=[('Temperature', 'temp_in__degC'), ('Gas Usage', 'g_use_cum_m3')],
    description='Data Type:',
    disabled=False)

# Locate the unique house IDs from the remeha dataframe
unique_ids = df_nfh_copy['id'].unique()

# Create the dropdown widget for selecting a House ID
id_dropdown = Dropdown(options=unique_ids, description='Select ID:')

# Update the plot so when a signal from the UI 
def update_plot(data_type, selected_id):
    plot_data(df_nfh_copy, df_weather_copy, data_type, selected_id)

# Use interactive to create a UI that reacts to the toggle button's state and the selected house ID
widgets.interactive(update_plot, data_type=data_type_selector, selected_id=id_dropdown)

In [ ]:
df_weather.head(40)